# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [1]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : d27c1c0
GPU    : Tesla T4
VRAM   : 14.6 GB


In [2]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 2.63s

7 passed · 0 warnings · 0 failures

Ready to submit.


In [5]:
# @title 3. Core pipeline — NB1 → NB5 (full eval, p95 length)
import os, re, json, subprocess, sys
from pathlib import Path
COMPUTE_TIER = "T4"
EVAL_LIMIT = ""
STAGES = "nb1 nb2 nb3 nb4 nb5"
os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ.pop("EVAL_LIMIT", None)
os.environ.pop("EPOCHS", None)
# NB1 measured p95=98, max=101; its suggested limit 256 covers every sample.
if not Path("results/token_stats.json").exists():
    subprocess.run([sys.executable, "scripts/colab_run.py", "nb1"], check=True)
stats = json.loads(Path("results/token_stats.json").read_text())
chosen_length = int(stats["suggested_max_length"])
assert chosen_length >= stats["max"]
os.environ["LAB_MAX_LENGTH"] = str(chosen_length)
Path("scripts/configure_length.py").write_text("from pathlib import Path\nimport re, sys, subprocess\nchosen = int(sys.argv[1])\nassert chosen > 0\ncfg = Path('src/labkit/config.py')\ns, n = re.subn(r'(name=\"T4\",[\\s\\S]*?max_length=)\\d+', lambda m: m[1]+'1024', cfg.read_text(), count=1)\nassert n == 1\ncfg.write_text(s)\nneedle = 'TIER = get_tier(os.environ.get(\"COMPUTE_TIER\", \"T4\"))'\nextension = '\\n# Experiment-only length override; keep hardware-tier defaults intact.\\nif os.environ.get(\"LAB_MAX_LENGTH\"):\\n    from dataclasses import replace\\n    TIER = replace(TIER, max_length=int(os.environ[\"LAB_MAX_LENGTH\"]))\\n'\nfor p in Path('notebooks').glob('*.py'):\n    source = p.read_text()\n    assert needle in source, p\n    if '# Experiment-only length override;' not in source:\n        p.write_text(source.replace(needle, needle + extension))\n    compile(p.read_text(), str(p), 'exec')\nsubprocess.run([sys.executable, 'scripts/build_colab.py'], check=True)\nprint('Default T4 remains 1024. Set LAB_MAX_LENGTH=' + str(chosen) + ' for the measured experiment.')\n")
Path("results/length_choice.json").write_text(json.dumps({"p95":stats["p95"],"max":stats["max"],"max_length":chosen_length,"reason":"NB1 measured distribution; covers all training samples"}, indent=2))
print("Chosen max_length:", chosen_length, "Full eval; default epochs")
# Stop only leftover lab workers from the interrupted initial download.
import psutil
repo_root = str(Path.cwd())
for worker in psutil.process_iter(["pid", "cmdline", "cwd"]):
    args = worker.info["cmdline"] or []
    if worker.pid != os.getpid() and worker.info["cwd"] == repo_root and any("notebooks/" in arg or "scripts/colab_run.py" in arg for arg in args[1:]):
        worker.terminate()
# Save complete qualitative evidence; leave all scorers unchanged.
nb5 = Path("notebooks/05_evaluate_and_verdict.py")
nb5_source = nb5.read_text()
marker = 'print("fine-tune:", scores_ft.as_dict())'
evidence_extension = '''
report.write_json([{"i": i, "ticket": row["input"], "label": row["label"], "prediction": pred, "ft_score": ev.triage_field_accuracy(pred, row["label"])} for i, (row, pred) in enumerate(zip(target, preds_ft))], "predictions_correct.json", results_dir=ROOT / "results")
report.write_json([{"i": i, "instruction": row["instruction"], "keywords": row["keywords"], "prediction": pred} for i, (row, pred) in enumerate(zip(regression, rpreds_ft))], "predictions_regression_correct.json", results_dir=ROOT / "results")
'''
assert marker in nb5_source
if "predictions_correct.json" not in nb5_source:
    nb5.write_text(nb5_source.replace(marker, marker + evidence_extension))
subprocess.run([sys.executable, "scripts/configure_length.py", str(chosen_length)], check=True)
print("Starting lab with live Colab logs")
!python -u scripts/colab_run.py {STAGES}


Chosen max_length: 256 Full eval; default epochs
Starting lab with live Colab logs
tier=T4  mask=assistant-only  eval_limit=full

NB1 — data, chat template & loss mask
tier=T4  model=unsloth/Qwen3.5-4B  max_length=256
250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh
eos_token: <|im_end|>
VERDICT: reasoning preserved — safe to train on traces

--- chuỗi đã render ---
<|im_start|>user
2+2?<|im_end|>
<|im_start|>assistant
<think>
buoc 1: kiem tra. buoc 2: tra loi.
</think>

4<|im_end|>

mode = assistant-only   supervised 39/94 (41%)
--- LOSS TÍNH TRÊN ĐOẠN NÀY ---
</think>

{"intent": "doi_tra", "urgency": "trung_binh", "pro

In [9]:
# @title 4. Gatekeeper + results + backup
import subprocess, sys, json, zipfile
from pathlib import Path
verification = subprocess.run([sys.executable, "scripts/verify.py"], capture_output=True, text=True)
print(verification.stdout)
print(verification.stderr)
for result_name in ["runs.csv", "verdict.json", "baselines_frozen.json"]:
    result_path = Path("results") / result_name
    print("\n---", result_name, "---")
    print(result_path.read_text() if result_path.exists() else "MISSING")
Path("results/verification_log.txt").write_text(verification.stdout + verification.stderr)
archive = Path("/content/Lab21_NguyenVuAnh_experiment_backup.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in ["results", "adapters/correct", "notebooks", "colab", "src", "scripts", "submission", "data"]:
        for item in Path(folder).rglob("*"):
            if item.is_file() and "__pycache__" not in item.parts:
                bundle.write(item, item.as_posix())
    for filename in ["requirements.txt", "README.md", "rubric.md", "Makefile"]:
        bundle.write(filename, filename)
print("Experiment backup:", archive, "bytes:", archive.stat().st_size)
print("Experiment backup created. Cell 6 builds the final submission package with the report.")
if verification.returncode != 0:
    print("Intermediate check only: finish report and use the strict final gate in cell 6.")



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 1.83s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[  ok  ] REPORT.md filled in                     

In [7]:
# @title 5. Frozen-prompt example replay + report review data
# Supplemental examples only; original baseline aggregate is never overwritten.
from pathlib import Path
import zipfile, json
Path("scripts/report_baseline_examples.py").write_text("import json, hashlib, pathlib, sys\nsys.path.insert(0, str(pathlib.Path.cwd() / 'src'))\nfrom labkit import generate, evaluate as ev, report\nfrom labkit.config import get_tier\nroot = pathlib.Path.cwd()\nfrozen = json.loads((root/'results/baselines_frozen.json').read_text())\nassert hashlib.sha256(generate.OPTIMIZED_PROMPT.encode()).hexdigest()[:16] == frozen['optimized_prompt_sha']\ntarget = [json.loads(line) for line in (root/'data/eval_target.jsonl').read_text().splitlines() if line.strip()]\nmodel, tok = generate.load_base(get_tier('T4'))\npreds, _ = generate.generate_batch(model, tok, [r['input'] for r in target], system=generate.OPTIMIZED_PROMPT, label='report-only frozen baseline examples')\nrows = [{'i':i, 'ticket':r['input'], 'label':r['label'], 'prediction':p, 'baseline_score':ev.triage_field_accuracy(p,r['label'])} for i,(r,p) in enumerate(zip(target,preds))]\nreport.write_json(rows,'predictions_baseline_b.json',results_dir=root/'results')\nprint('Report-only baseline replay target:',sum(r['baseline_score'] for r in rows)/len(rows))\nprint('Original frozen aggregate remains unchanged:',frozen['baseline_b']['target'])\nregression = [json.loads(line) for line in (root/'data/eval_regression.jsonl').read_text().splitlines() if line.strip()]\nrpreds, _ = generate.generate_batch(model, tok, [r['instruction'] for r in regression], system=None, max_new_tokens=96, label='report-only frozen regression examples')\nrrows = [{'i':i, 'instruction':r['instruction'], 'keywords':r['keywords'], 'prediction':p, 'baseline_score':ev.keyword_recall(p,r['keywords'])} for i,(r,p) in enumerate(zip(regression,rpreds))]\nreport.write_json(rrows,'predictions_regression_baseline_b.json',results_dir=root/'results')\nprint('Report-only baseline replay regression:',sum(r['baseline_score'] for r in rrows)/len(rrows))\n")
!python -u scripts/report_baseline_examples.py
assert len(json.loads(Path("results/predictions_baseline_b.json").read_text())) == 50
assert len(json.loads(Path("results/predictions_regression_baseline_b.json").read_text())) == 15
review_zip = Path("/content/Lab21_review_results.zip")
with zipfile.ZipFile(review_zip, "w", zipfile.ZIP_DEFLATED) as bundle:
    for item in Path("results").rglob("*"):
        if item.is_file():
            bundle.write(item, item.as_posix())
print("Report review data:", review_zip)
from google.colab import files
files.download(str(review_zip))


Loading weights: 100% 426/426 [00:30<00:00, 14.01it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
  [report-only frozen b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# @title 6. Final report + strict verification + submission ZIP
from pathlib import Path
import json, hashlib, subprocess, sys, zipfile
# The report is a record of the completed run. New results need a new report.
expected_hashes = {"baselines_frozen.json":"9e62ea9892e914d8b1bb7bb8c4666ae02a72e92dbb62b3c74be260075b54e6f2","runs.csv":"4418defeb6e43642a72602e25ad6d0728f018d0cefe5267404411634af4d2328","verdict.json":"2859807a16e95ef5ddc05b73eecd7bacdb67159bd9946beaabc4b154902e98ca","autopsy.json":"e1aa7ee65e76cb2a1461c1b5400496a415050028e693121a27162289ca15fe8f","mask_proof.json":"a7d94d282e36b594dd2d8045e6b278844fc2f5bd1bf4ca71831bc73dd09dc910","token_stats.json":"8bd9c33599d01dd5ba185264b40b516fa39b57f79cee2378284f47e37667532b","predictions_correct.json":"25a4a03c90c5b375286ec0bf6636001b4dc1071ea9dbb1eaf8db9e7250f95287","predictions_baseline_b.json":"6f209443e0b69e0d96ed55198a6a56ad36c86b0345dff09adc7c6e8ce784c605","predictions_regression_correct.json":"76f6ca5a402ab2dcaae9679a9fd0834df52f9de9d1284f48811e6b2bde54626d","predictions_regression_baseline_b.json":"c81a46efbdda17da7d924a66515c063ff46edd11defe53f62361bc34613f2d25"}
for name, expected in expected_hashes.items():
    actual = hashlib.sha256((Path("results") / name).read_bytes()).hexdigest()
    assert actual == expected, "Results changed: update report from actual new results before packaging: " + name
Path("submission/REPORT.md").write_text("# Lab 21 — LoRA fine-tuning và đánh giá đối chứng\n\n**Học viên:** Nguyễn Vũ Anh  \n**Mã học viên:** 2A202602502  \n**Lớp:** Track 3: AI Application\n\n## Phạm vi và lựa chọn\n\nThí nghiệm sử dụng `unsloth/Qwen3.5-4B` theo cấu hình T4 của bộ lab, với tác vụ phân loại ticket chăm sóc khách hàng tiếng Việt thành bốn trường JSON: intent, urgency, product, sentiment. Tác vụ này cho phép đo trực tiếp độ đúng của từng trường và tách lỗi định dạng khỏi lỗi phân loại. Dùng cùng base model và dữ liệu cho baseline và tất cả adapter để so sánh có ý nghĩa.\n\nMôi trường thực thi: Google Colab, Tesla T4 14,6 GB, fp16. Mã nguồn của lab ở commit `d27c1c0`. Tập dữ liệu gồm 250 mẫu; chia 225 train/25 validation với seed 42. Đánh giá đầy đủ trên 50 ticket mục tiêu và 15 câu hỏi regression. Không đặt `EVAL_LIMIT`, không giảm epoch mặc định.\n\n## Mask và độ dài chuỗi\n\nNB1 xác nhận assistant-only loss: ví dụ kiểm tra có 39/94 token được giám sát, tỷ lệ 0,4149; `answer_is_supervised=true`, `question_is_masked=true`. Trên tập train, 9014/20951 token được giám sát (43,0%). Vì vậy loss tập trung vào câu trả lời thay vì học lặp lại câu hỏi.\n\nPhân bố 250 mẫu: mean 93,1; p50=93; p95=98; p99=100; max=101 token. Chọn `max_length=256`, theo gợi ý thực đo của NB1 và đủ chứa mọi mẫu. Lần đo dùng giới hạn 256 cho cả bốn cấu hình. Notebook tái lập áp dụng LAB_MAX_LENGTH=256 bằng dataclasses.replace trên TIER riêng của từng stage, giữ cấu hình phần cứng mặc định 1024 của bộ lab. Các tham số thực đo nằm trong runs.csv và length_choice.json. NB1 kiểm tra chat template giữ phần reasoning.\n\n## Baseline đóng băng trước huấn luyện\n\n| Cấu hình | Target field accuracy | Regression keyword recall | JSON format | Latency ms/mẫu |\n|---|---:|---:|---:|---:|\n| (a) Base + naive prompt | 0,000 | 0,791111 | 0,000 | 3173,836 |\n| (b) Base + optimized prompt | 0,765 | 0,791111 | 1,000 | 1036,796 |\n\nBaseline (b) tốt hơn (a) trước khi fine-tuning. Các số liệu được lưu trong `results/baselines_frozen.json`; SHA của optimized prompt là `719e74d3b6232053`. Target là trung bình độ đúng của bốn trường, không phải tỷ lệ ticket đúng toàn bộ. Regression đo keyword recall, không khẳng định đánh giá đầy đủ khả năng tổng quát của model.\n\n## Huấn luyện và đối chứng\n\nDùng cùng seed 42, dữ liệu train 225 mẫu, assistant-only mask, max_length=256, batch=1 và gradient_accumulation=16 (effective batch 16, dưới 32). Cấu hình chuẩn có all text-linear, rank 16, alpha 32, LR=1e-4, bằng 10 lần mức full-FT 1e-5 dùng trong đối chứng. Scheduler cosine, warmup 3 bước, gradient checkpointing. Ngân sách mặc định là 2 epoch, 30 bước theo scheduler. Attention-only dùng q/v, tăng rank lên 283 và alpha 566 để khớp số tham số; chênh 8192 tham số, khoảng 0,02523%, dưới 5%. QLoRA dùng base 4-bit cả khi train và khi chấm adapter.\n\n| Run | Placement | r / alpha | LR | Base | Trainable params | Train_loss trung bình | Train giây | VRAM GB | Bước |\n|---|---|---:|---:|---|---:|---:|---:|---:|---:|\n| correct | text-linear | 16 / 32 | 0.0001 | fp16 | 32464896 | 0.6258 | 394.7 | 8.78 | 30 |\n| attn_only | attn-only | 283 / 566 | 0.0001 | fp16 | 32456704 | 0.5386 | 255.9 | 8.79 | 30 |\n| wrong_lr | text-linear | 16 / 32 | 1e-05 | fp16 | 32464896 | 1.5702 | 388.5 | 8.78 | 30 |\n| qlora | text-linear | 16 / 32 | 0.0001 | 4-bit | 32464896 | 0.7058 | 466.5 | 3.86 | 30 |\n\nCột final_loss trong runs.csv do bộ lab ghi train_loss tổng hợp trung bình, không phải loss của riêng batch cuối. Toàn bộ pipeline NB1–NB5 mất khoảng 3125 giây (52,1 phút), gồm cả tải model, nạp trọng số và đánh giá. Log có một số grad_norm=NaN ở fp16; kiểm tra toàn bộ bốn file adapter cho thấy không có NaN/Inf và các tham số đều khác 0 (results/adapter_finite_check.json).\n\n## Bốn nhóm đánh giá và giải phẫu đối chứng\n\nBaseline (b) dùng OPTIMIZED_PROMPT đã chốt; FT dùng NAIVE_PROMPT ngắn theo thiết kế NB5. Các tập đánh giá và SHA prompt vẫn khớp bản đóng băng. Target là độ đúng trung bình của 4 trường. Format là tỷ lệ khóa bắt buộc hiện diện qua bộ parse JSON linh hoạt của lab; điểm 1,0 không tự chứng minh chỉ có JSON hoặc mọi giá trị đúng vocabulary. Regression dùng keyword recall, chuẩn hóa tiếng Việt giống nhau ở prediction và keyword. Latency dùng kết quả ms/mẫu của cùng hàm chấm.\n\n| Run | Target | Regression | Format | Latency ms/mẫu | n target |\n|---|---:|---:|---:|---:|---:|\n| (a) base + naive prompt | 0 | 0.7911 | 0 | 3173.8 | 50 |\n| (b) base + optimized prompt | 0.765 | 0.7911 | 1 | 1036.8 | 50 |\n| (c) LoRA fine-tune | 0.98 | 0.5222 | 1 | 1352.4 | 50 |\n\n| Run | Target | Format | Latency ms/mẫu | n |\n|---|---:|---:|---:|---:|\n| correct | 0.98 | 1 | 1352.4 | 50 |\n| attn_only | 0.965 | 1 | 885.9 | 50 |\n| wrong_lr | 0 | 0 | 5170.1 | 50 |\n| qlora | 0.94 | 1 | 1761.1 | 50 |\n\nNhóm regression được đo cho (a), (b) và correct để tính gate bốn nhóm. Theo NB5, ba đối chứng được chấm target/format/latency; không có số regression của các đối chứng trong bảng này. Xếp hạng theo target: correct > attn_only > qlora > wrong_lr. Attention-only có train_loss thấp hơn correct nhưng target thấp hơn 1,5 điểm phần trăm; thứ tự theo loss khác thứ tự theo tác vụ. Wrong_lr không tạo được JSON đủ khóa trong đánh giá này nên target và format đều bằng 0. QLoRA giảm VRAM khoảng 56% nhưng target thấp hơn 4 điểm phần trăm và latency cao hơn correct khoảng 30%; đây là đánh đổi thực đo của môi trường này. Chỉ số valid_trace_rate của correct là 0,0. Tác vụ train chủ yếu là JSON; chỉ số này riêng nó chưa đủ đánh giá năng lực suy luận tổng quát. Kiểm tra template ở NB1 là kiểm tra giữ cấu trúc tag, không phải chứng minh chất lượng reasoning.\n\n## Ví dụ định tính: cả thắng, sai và thua\n\nĐối chiếu đầy đủ 50 ticket: 34 ca FT thắng, 16 ca hòa, 0 ca thua baseline (b) theo field accuracy. FT còn 4 ticket sai urgency (i=3,12,41,46); 46/50 ticket đúng cả bốn trường, trong khi target=0,98 là 196/200 trường đúng. Ba ca tệ nhất NB5 (i=3,12,41) đều được đưa dưới đây. Vì không có ca thua baseline trong nhóm target, hai ca thua thật được thể hiện ở regression. Trong 15 câu regression có 1 ca thắng, 7 ca thua và 7 ca hòa. Không đổi dữ liệu hay điểm số để tạo ca thua.\n\nVí dụ baseline được tái sinh bổ sung với đúng model và prompt đã đóng băng để có dự đoán từng mẫu; điểm tái sinh target=0,765 và regression=0,791111 khớp baseline gốc. Các aggregate baseline gốc và verdict gốc được giữ nguyên.\n\n### Target i=0\n\n**Ticket:** Cho mình hỏi, mình đặt chuột không dây mã đơn VN232232. Cho tôi trả lại. Gấp. Shop hỗ trợ tốt.\n\n**Nhãn:** `{\"intent\":\"doi_tra\",\"urgency\":\"cao\",\"product\":\"chuột không dây\",\"sentiment\":\"tich_cuc\"}`\n\n**Fine-tune:** `{\"intent\": \"doi_tra\", \"urgency\": \"cao\", \"product\": \"chuột không dây\", \"sentiment\": \"tich_cuc\"}`\n\n**Baseline (b):** `{\"intent\": \"hoan_tien\", \"urgency\": \"cao\", \"product\": \"chuột không dây\", \"sentiment\": \"tich_cuc\"}`\n\nBaseline nhầm intent hoan_tien; nhãn và FT là doi_tra. Các trường còn lại đúng. Đây là ca FT thắng 1,00 so với 0,75.\n\n### Target i=1\n\n**Ticket:** Shop ơi, mình đặt ốp lưng điện thoại mã đơn VN812931. Hoàn tiền. Sớm nhé. Bực mình.\n\n**Nhãn:** `{\"intent\":\"hoan_tien\",\"urgency\":\"trung_binh\",\"product\":\"ốp lưng điện thoại\",\"sentiment\":\"tieu_cuc\"}`\n\n**Fine-tune:** `{\"intent\": \"hoan_tien\", \"urgency\": \"trung_binh\", \"product\": \"ốp lưng điện thoại\", \"sentiment\": \"tieu_cuc\"}`\n\n**Baseline (b):** `{\"intent\": \"hoan_tien\", \"urgency\": \"cao\", \"product\": \"ốp lưng điện thoại\", \"sentiment\": \"tieu_cuc\"}`\n\nBaseline dự đoán urgency=cao cho “Sớm nhé”; nhãn và FT là trung_binh. FT thắng 1,00 so với 0,75.\n\n### Target i=3\n\n**Ticket:** Cho mình hỏi, mình đặt bình giữ nhiệt mã đơn VN804124. Chưa thấy tiền. Khi nào tiện. Cảm ơn shop nhiều.\n\n**Nhãn:** `{\"intent\":\"hoan_tien\",\"urgency\":\"thap\",\"product\":\"bình giữ nhiệt\",\"sentiment\":\"tich_cuc\"}`\n\n**Fine-tune:** `{\"intent\": \"hoan_tien\", \"urgency\": \"trung_binh\", \"product\": \"bình giữ nhiệt\", \"sentiment\": \"tich_cuc\"}`\n\n**Baseline (b):** `{\"intent\": \"hoan_tien\", \"urgency\": \"trung_binh\", \"product\": \"bình giữ nhiệt\", \"sentiment\": \"tich_cuc\"}`\n\n“Khi nào tiện” có nhãn urgency=thap, nhưng cả hai mô hình dự đoán trung_binh. FT sai một trường và hòa baseline ở 0,75.\n\n### Target i=12\n\n**Ticket:** Shop ơi, mình đặt áo khoác gió mã đơn VN613097. Bị lỗi. Khi nào tiện. Cảm ơn shop nhiều.\n\n**Nhãn:** `{\"intent\":\"san_pham_loi\",\"urgency\":\"thap\",\"product\":\"áo khoác gió\",\"sentiment\":\"tich_cuc\"}`\n\n**Fine-tune:** `{\"intent\": \"san_pham_loi\", \"urgency\": \"trung_binh\", \"product\": \"áo khoác gió\", \"sentiment\": \"tich_cuc\"}`\n\n**Baseline (b):** `{\"intent\": \"san_pham_loi\", \"urgency\": \"trung_binh\", \"product\": \"áo khoác gió\", \"sentiment\": \"tich_cuc\"}`\n\nTicket sản phẩm lỗi kèm “Khi nào tiện” vẫn bị gán urgency=trung_binh. Nhãn là thap; cả FT và baseline đạt 0,75.\n\n### Target i=41\n\n**Ticket:** Cho mình hỏi, mình đặt đèn bàn LED mã đơn OD436045. Giao hàng chậm. Khi nào tiện. Cảm ơn shop nhiều.\n\n**Nhãn:** `{\"intent\":\"van_chuyen\",\"urgency\":\"thap\",\"product\":\"đèn bàn LED\",\"sentiment\":\"tich_cuc\"}`\n\n**Fine-tune:** `{\"intent\": \"van_chuyen\", \"urgency\": \"trung_binh\", \"product\": \"đèn bàn LED\", \"sentiment\": \"tich_cuc\"}`\n\n**Baseline (b):** `{\"intent\": \"hoi_thong_tin\", \"urgency\": \"trung_binh\", \"product\": \"đèn bàn LED\", \"sentiment\": \"tich_cuc\"}`\n\nFT sửa được intent từ hoi_thong_tin của baseline thành van_chuyen, nhưng urgency vẫn là trung_binh thay vì thap. FT chưa đúng hết: 0,75 so với baseline 0,50.\n\n### Regression i=2 — FT thua\n\n**Câu hỏi:** 1 km bằng bao nhiêu mét?\n\n**Keyword đúng:** `1000`. Baseline keyword recall=1; FT=0.\n\n**Fine-tune:** `{\"intent\": \"hoi_thong_tin\", \"urgency\": \"thap\", \"product\": null, \"sentiment\": \"trung_tinh\"}`\n\n**Baseline (b), trích đoạn:** “Kết quả: 1 km = 1000 m.”\n\nFT áp dụng schema phân loại ticket lên câu hỏi kiến thức thông thường, không trả lời giá trị cần thiết. Đây là lỗi hành vi trên regression, dù JSON có vẻ hợp lệ. Dự đoán đầy đủ của cả hai được giữ trong results/predictions_regression_*.json.\n\n### Regression i=9 — FT thua\n\n**Câu hỏi:** Một năm có bao nhiêu tháng?\n\n**Keyword đúng:** `12`. Baseline keyword recall=1; FT=0.\n\n**Fine-tune:** `{\"intent\": \"hoi_thong_tin\", \"urgency\": \"thap\", \"product\": null, \"sentiment\": \"trung_tinh\"}`\n\n**Baseline (b), trích đoạn:** “Một năm bình thường có 12 tháng.”\n\nFT áp dụng schema phân loại ticket lên câu hỏi kiến thức thông thường, không trả lời giá trị cần thiết. Đây là lỗi hành vi trên regression, dù JSON có vẻ hợp lệ. Dự đoán đầy đủ của cả hai được giữ trong results/predictions_regression_*.json.\n\n## Phân tích verdict (dựa trên NB5 đã đo)\n\nVerdict là FAIL vì regression giảm 0,268889, vượt xa mức giảm cho phép 0,02, dù target tăng 0,215 so với baseline tối ưu. Cấu hình chuẩn đạt 0,98 về độ đúng trung bình của bốn trường; đây là cải thiện rõ trên tập 50 ticket này. Tuy nhiên, kết luận triển khai phải xét cả hai điều kiện của gate. Chỉ so với baseline ngây thơ có target bằng 0 sẽ che mất yêu cầu quan trọng: đối thủ thực sự là baseline đã tối ưu trước huấn luyện. Điểm format bằng 1,0 cũng không bù được việc trả lời kém hơn trên các câu hỏi regression. Latency của fine-tune khoảng 1352,4 ms/mẫu, tăng khoảng 30,4% so với 1036,8 ms/mẫu của baseline (b), dù dùng prompt ngắn hơn. Kết quả này chưa đủ lý do thay baseline bằng adapter chuẩn cho một trợ lý cần giữ năng lực rộng. Một bước tiếp theo có thể thử là thêm 1–5% replay dữ liệu tổng quát và đánh giá lại bằng gate giữ nguyên; thí nghiệm replay chưa được thực hiện trong kết quả core này. Regression ở đây là keyword recall trên 15 câu hỏi, nên diễn giải là suy giảm trên bộ kiểm tra này, chưa phải phép đo toàn diện mọi năng lực của model.\n\n## Bài học và giới hạn rút ra\n\nBài học đầu tiên là cần chốt baseline và tập đánh giá trước khi nhìn kết quả fine-tuning. Prompt tối ưu đã nâng target từ 0 lên 0,765 mà chưa huấn luyện, cho thấy prompt engineering là một đối thủ đáng kể. Bài học thứ hai là phải kiểm chứng loss mask và độ dài chuỗi bằng dữ liệu. Assistant-only mask được kiểm tra trực tiếp, còn giới hạn 256 token được chọn sau khi đo p95=98 và max=101, thay vì dùng một số lớn theo thói quen. Bài học thứ ba là loss huấn luyện không quyết định chất lượng ngoài tập train: attention-only có train_loss trung bình 0,5386, thấp hơn 0,6258 của cấu hình chuẩn, nhưng target lại thấp hơn (0,965 so với 0,98). Vì thế, xếp hạng bằng loss sẽ cho thứ tự khác xếp hạng bằng tác vụ. Bài học thứ tư là tiết kiệm VRAM chưa đồng nghĩa tiết kiệm thời gian: QLoRA dùng 3,86 GB nhưng thời gian train dài hơn cấu hình fp16 trong môi trường này. Kết quả chỉ đại diện một seed, một tập dữ liệu nhỏ và một GPU. Chưa có nhiều lần chạy để ước lượng độ biến thiên hoặc khoảng tin cậy. Log fp16 có gradient NaN xen kẽ; các trọng số đã lưu đều hữu hạn, nhưng chưa ghi số cập nhật thực bị GradScaler bỏ qua. Do đó, bảng 30 bước thể hiện ngân sách bước theo scheduler của bộ lab, chưa chứng minh mọi cấu hình có đúng 30 cập nhật hữu hiệu. Đây là giới hạn phải giữ khi diễn giải chênh lệch nhỏ giữa các cấu hình.\r\n\n\n## Tái lập và các file bằng chứng\n\nMã nguồn: [VinUni Lab21](https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab), commit d27c1c02ebe99f32f52f706be88b4c30fb1d7fca. [Notebook thực thi](https://colab.research.google.com/drive/1atokmxT0U-Qd0LXFAm7rIXAphLw4fSdm) dùng T4 và full eval. Python 3.13.15; torch 2.11.0+cu130; transformers 5.18.0; TRL 1.14.2. Danh sách phiên bản đầy đủ trong results/environment.json.\n\nChạy các ô từ trên xuống trong notebook đã lưu. Ô pipeline đo token_stats khi cần, đặt LAB_MAX_LENGTH=256 theo dữ liệu, áp dụng giới hạn riêng trên TIER, xuất dự đoán và tạo lại colab/*.ipynb từ notebooks/*.py để đồng bộ. Khi tái chạy từ archive bằng terminal, đặt COMPUTE_TIER=T4 và LAB_MAX_LENGTH=256; bỏ EVAL_LIMIT và EPOCHS để dùng full eval và mặc định 2 epoch; chạy python scripts/colab_run.py nb1 nb2 nb3 nb4 nb5, rồi python scripts/verify.py.\n\nBằng chứng chính: template_check.json, mask_proof.json, token_stats.json, length_choice.json, baselines_frozen.json, runs.csv, verdict.json, autopsy.json, qualitative.json, predictions_correct.json, predictions_baseline_b.json và các predictions_regression_*.json. Log kiểm tra nộp bài cuối lưu ở results/verification_log.txt. Gói nộp gồm report, results, adapter correct, notebook thực thi và mã nguồn/notebook cần để tái lập.\n\nReport ghi nhận lần chạy này. Khi huấn luyện lại, cập nhật bảng số và ví dụ từ kết quả mới trước khi đóng gói; ô final kiểm tra hash các dữ liệu đầu vào của report để ngăn dùng nhầm report cũ.\n", encoding="utf-8")
Path("scripts/check_adapter_finite.py").write_text("from pathlib import Path\nimport json, torch\nfrom safetensors import safe_open\nrows=[]\nfor folder in sorted(Path('adapters').iterdir()):\n    for path in folder.glob('*.safetensors'):\n        n=nonfinite=nonzero=0\n        with safe_open(str(path), framework='pt', device='cpu') as f:\n            for key in f.keys():\n                tensor=f.get_tensor(key)\n                n+=tensor.numel()\n                nonfinite+=int((~torch.isfinite(tensor)).sum())\n                nonzero+=int(torch.count_nonzero(tensor))\n        rows.append({'run':folder.name,'file':str(path),'elements':n,'nonfinite_elements':nonfinite,'nonzero_elements':nonzero})\nprint(json.dumps(rows,indent=2))\nPath('results/adapter_finite_check.json').write_text(json.dumps(rows,indent=2))\nassert rows and all(row['nonfinite_elements']==0 for row in rows)\n")
subprocess.run([sys.executable, "scripts/check_adapter_finite.py"], check=True)
verification = subprocess.run([sys.executable, "scripts/verify.py"], capture_output=True, text=True)
print(verification.stdout)
print(verification.stderr)
Path("results/verification_log.txt").write_text(verification.stdout + verification.stderr)
assert verification.returncode == 0, "Final gate failed: fix the FAIL entries before submission"
archive = Path("/content/Lab21_NguyenVuAnh_2A202602502.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in ["results", "adapters/correct", "notebooks", "colab", "src", "scripts", "submission", "data", "tests"]:
        for item in Path(folder).rglob("*"):
            if item.is_file() and "__pycache__" not in item.parts:
                bundle.write(item, item.as_posix())
    for filename in ["requirements.txt", "README.md", "rubric.md", "Makefile", "pyproject.toml", "pytest.ini", "setup.cfg"]:
        if Path(filename).is_file():
            bundle.write(filename, filename)
print("Submission package:", archive, "bytes:", archive.stat().st_size)
print("Experiment verdict remains FAIL due to regression; artifact gate passed.")
from google.colab import files
files.download(str(archive))



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 1.85s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[  ok  ] REPORT.md filled in                     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Bonus B5 publication
Public adapter: https://huggingface.co/vustaz/Lab21-Qwen3.5-4B-LoRA

Updated submission report: https://github.com/vuanh259/Lab21-LoRA-2A202602502/blob/main/submission/REPORT.md
The original executed outputs are preserved. Bonus publication does not change experiment metrics or the FAIL verdict.